In [0]:
# =========================================================================
# GOLD LAYER: dim_territories Worker Notebook
# =========================================================================

GOLD_STORAGE_PATH = "abfss://gold@aze2estorage.dfs.core.windows.net/dimensions/dim_territories"
TABLE_NAME = "gold.dim_territories"

print(f"🔄 Starting build for {TABLE_NAME}...")
print(f"📂 Target Storage Path: {GOLD_STORAGE_PATH}")

# 1. Read Silver Territories Table
df_territories = df_territories = spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/territories")
print(f"📥 Read {df_territories.count()} records from silver.territories")

# 2. Select and format columns, aliasing the key for clean Star Schema joins
gold_dim_territories = df_territories.select(
    df_territories["sales_territory_key"].alias("territory_key"),
    df_territories["region"],
    df_territories["country"],
    df_territories["continent"]
)

# 3. Write Delta files to explicit Gold storage path
gold_dim_territories.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_STORAGE_PATH)

print("💾 Successfully wrote Delta files to storage path.")

# 4. Register/Update External Table in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_STORAGE_PATH}'
""")

print(f"🔗 Registered external table: {TABLE_NAME}")

# 5. Performance Optimization
spark.sql(f"OPTIMIZE {TABLE_NAME} ZORDER BY (territory_key)")

print(f"⚡ Successfully optimized {TABLE_NAME} with Z-Ordering.")
print(f"🎉 {TABLE_NAME} build complete!")